In [1]:
#!/usr/bin/env python
# coding: utf-8


import h5py
import tabulate
import contextily as ctx
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
import random
import math
from datetime import datetime, timedelta
# from IPython.display import HTML, display
from os import path
from shapely.geometry import Point
from scipy import stats
import time
import multiprocessing
from multiprocessing import Pool
from rasterio.plot import show
import functools
from osgeo import ogr
from osgeo import gdal
import os
from glob import glob
from os import path
import rasterio as rio
from rasterio.mask import mask
from rasterio.merge import merge
from rasterio.io import MemoryFile
from joblib import Parallel, delayed
from sklearn.base import clone
from joblib import load
import pickle
from affine import Affine
import json


cwd = os.getcwd()
print(cwd)

##filling in the argument lists 
import glob
import re

pattern2  ='re_xgboost_sqrt_nbr_lhs_33_dodoma2+2sig_sen97_beam+sol_nbr' 

###locating model and related predictor names, back_trans bias correction C###
f=pd.read_csv('agbd_predictions/rev_dodoma_ts_out/model_obj/re_xgboost_sqrt_nbr_lhs_33_dodoma2+2sig_sen97_beam+sol_nbr_model_Features.csv', header=None) ###!!! need to change!!!###
features = np.array(f.iloc[:,0]).tolist()
print('features',len(features), features)
fm ='agbd_predictions/rev_dodoma_ts_out/model_obj/re_xgboost_sqrt_nbr_lhs_33_dodoma2+2sig_sen97_beam+sol_nbr_model.sav' ###!!! need to change!!!###
Cf= pd.read_csv('agbd_predictions/rev_dodoma_ts_out/model_obj/re_xgboost_sqrt_nbr_lhs_33_dodoma2+2sig_sen97_beam+sol_nbr_model_C.csv', header=None) ###!!! need to change!!!###
C = Cf.iloc[:,0][0]
print(C)

response_var=pd.read_csv('agbd_predictions/rev_dodoma_ts_out/model_obj/re_xgboost_sqrt_nbr_lhs_33_dodoma2+2sig_sen97_beam+sol_nbr_model_train_response_var.csv', header=None)
response_se=pd.read_csv('agbd_predictions/rev_dodoma_ts_out/model_obj/re_xgboost_sqrt_nbr_lhs_33_dodoma2+2sig_sen97_beam+sol_nbr_model_train_response_se.csv', header=None)
alltraining=np.load('agbd_predictions/rev_dodoma_ts_out/model_obj/re_xgboost_sqrt_nbr_lhs_33_dodoma2+2sig_sen97_beam+sol_nbr_model_train_alltraining.npy')
print(alltraining.shape)



#########
args=[]
nboot=100
trans_agbd = 'sqrt'
years = [2021] 
pheno='lhs'
t1='168064'   #tile name
t2=''         #tile name 2, optional
ts=[t1]
datev='1018'   
mod = pickle.load(open(fm, 'rb')) 
iccs =[False]  #whether explicit AGC interannual harmonization is needed
best_params='NULL'
#Landsat tiles downloaded from GEE were separated in these sub-tiles; here mosaic the ones overlapping with study areas
tif_files2 = ['0000005120-0000000000','0000005120-0000002560','0000005120-0000005120','0000002560-0000002560','0000002560-0000005120'] ###!!! need to change!!!##
tif_files1 = [] 

# Create the folder if it doesn't exist
dst_dir= 'agbd_predictions/rev_dodoma64_ts_out_pred/'
os.makedirs('agbd_predictions/rev_dodoma64_ts_out_pred/', exist_ok=True) ###!!! need to change!!!###

tif_paths=[]
##########


for i in years:
    for t in ts:
        yPred = i 
        print(i)
        fitInd='NBR_3by3_dur8_biome1-'
        if t == '173060':
            if i < 2019:
                if i <= 2015:
    #                 print(i)
                    fitInd='NBR_3by3_dur8_biome1_val_v2-'  #+'-'+corner+'.tif'
                elif i>2015: 
                    print('no')
                    fitInd='NBR_3by3_dur8_biome1_val-'  #+'-'+corner+'.tif'
        else:
            fitInd='NBR_3by3_dur8_biome1_val_v2-'  #+'-'+corner+'.tif'


        imgPattern ='ls_'+pheno+'_medoid_'+t+'_'+str(yPred) +'_'+fitInd + '*.tif' 
        print(imgPattern)    
        if t =='173060':
            t_folder = 'kibale_2000-2021'
        else:
            t_folder = 't'+t+'_1986-2021'

        pdir= ".../EA_data/landsat_comp/"+ t_folder+"/"+ str(yPred)   #EA_data/gd_data/

        
        if t == '168051':
            
            for c in tif_files1:
                path= pdir +"/"+'ls_'+pheno+'_medoid_'+t+'_'+str(yPred)+'_'+fitInd+c+'.tif' 
                #'ls_'+pheno+'_medoid_'+t+'_'+yPred+'_NBR_3by3_dur8_biome1-'+c+'.tif'
                tif_paths.append(path)
        else:
            for c in tif_files2:
                path= pdir +"/"+'ls_'+pheno+'_medoid_'+t+'_'+str(yPred)+'_'+fitInd+c+'.tif' 
                #'ls_'+pheno+'_medoid_'+t+'_'+yPred+'_NBR_3by3_dur8_biome1-'+c+'.tif'
                tif_paths.append(path)

    print(tif_paths)
    dst= 'agbd_'+str(yPred)+'_'+pattern2+'_'+str(len(ts))+'tiles_'+datev+'.tif'  
    print(dst)
    for icc in iccs: 
        arg = nboot, response_var,response_se, alltraining, yPred, t1, tif_paths, mod, features, dst_dir, dst, icc, best_params
        args.append(arg)


def back_trans(x):
    x = np.squeeze(x)
    if trans_agbd == 'sqrt':
        return np.square(x)
    else:
        return np.exp(x)


def get_band_indices_by_name(dataset, target_band_names):
    band_names = list(dataset.descriptions)  # 0-based
    return [i + 1 for i, name in enumerate(band_names) if name in target_band_names]


def read_selected_bands(path, target_band_names):
    src = rasterio.open(path)
    indices = get_band_indices_by_name(src, target_band_names)
    band_data = src.read(indexes=indices)  # shape: (bands, height, width)
    selected_names= [src.descriptions[i - 1] for i in indices]
    crs = src.crs
    return band_data, src, selected_names, crs


def merge_selected_bands_in_memory(tif_paths, sel_bands): #both are lists

    raster_paths = tif_paths  # etc.
    target_band_names = sel_bands 

    # Collect all (dataset, selected_band_indices)
    datasets = []
    band_arrays = []
    band_names=[]

    for path in raster_paths:
        data, src, sel_names, crs = read_selected_bands(path, target_band_names)
        datasets.append(src)
        band_arrays.append(data)
        band_names.append(sel_names)

    # Reopen subsets as temporary datasets
    subset_datasets = []

    for data, src in zip(band_arrays, datasets):
        meta = src.meta.copy()
        meta.update({
            "count": data.shape[0],
            "height": data.shape[1],
            "width": data.shape[2],
            "transform": src.transform,
            "driver": "GTiff"
        })
        memfile = MemoryFile()
        mem = memfile.open(**meta)
        mem.write(data)
        subset_datasets.append(mem)

    merged_array, merged_transform = merge(subset_datasets, method='first')

    return merged_array, merged_transform, band_names[0], crs




#for reach tile and each model, this only needs to be done onces
def train_and_save_ensemble(nboot, reference_y, reference_se, xtraining,year, tile,tif_paths, model, features, dst_dir,dst_f, icc=True, best_params=None): 
    
    n_bootstrap = nboot
    reference_y = np.array(reference_y)
    reference_se = np.array(reference_se)
    sampled_y = np.random.normal(reference_y, reference_se, size=( len(reference_se),n_bootstrap))  #!!double check the length, to see if it's the same as the length of the df

    # Replace negative values with zero
    sampled_y[sampled_y < 0] = 0  #in case the sampling sampled to the negative values

    if trans_agbd!='sqrt':
        sampled_y=np.log(sampled_y)  #transformed the sampled ref data 
    else: 
        sampled_y=np.sqrt(sampled_y)
    
    models = []
    for i in range(n_bootstrap):
        print(i)

        sampled_y_column= sampled_y[:, i]

        # Create a new model instance each time
        model_i =  clone(model)


        if best_params != 'NULL':
            #model = model #using the best model tuned and var_selected  #xgb.XGBRegressor( tree_method='hist', early_stopping_rounds=20, random_state=42)
            model_i.fit(xtraining, sampled_y_column, **best_params)  #to reflect the correct model, need to change x
        else:
            model_i.fit(xtraining, sampled_y_column)

        models.append(model_i)
        
    from joblib import dump
    output_dir='agbd_predictions/rev_pred_cache/ensemble_model/' 
    dump(models, output_dir+'ls_lhs_medoid_'+tile+'_NBR_3by3_dur8_biome1_val_v2_100EnsembleModels.joblib')




def merge_agbd_export(nboot, reference_y, reference_se, xtraining,year, tile,tif_paths, model, features, dst_dir,dst_f, icc=True, best_params=None): 

    si =set(['ftv_b1_fit_contrast', 'ftv_b1_fit_diss', 'ftv_b1_fit_savg', 'ftv_b1_fit_idm', 'ftv_b1_fit_asm', 
     'ftv_b1_fit_ent', 'ftv_b1_fit_var', 'ftv_b1_fit_corr', 'ftv_b2_fit_contrast', 'ftv_b2_fit_diss', 
     'ftv_b2_fit_savg', 'ftv_b2_fit_idm', 'ftv_b2_fit_asm', 'ftv_b2_fit_ent', 'ftv_b2_fit_var', 'ftv_b2_fit_corr', 
     'ftv_b3_fit_contrast', 'ftv_b3_fit_diss', 'ftv_b3_fit_savg', 'ftv_b3_fit_idm', 'ftv_b3_fit_asm', 
     'ftv_b3_fit_ent', 'ftv_b3_fit_var', 'ftv_b3_fit_corr', 'ftv_b4_fit_contrast', 'ftv_b4_fit_diss', 
     'ftv_b4_fit_savg', 'ftv_b4_fit_idm', 'ftv_b4_fit_asm', 'ftv_b4_fit_ent', 'ftv_b4_fit_var', 'ftv_b4_fit_corr', 
     'ftv_b5_fit_contrast', 'ftv_b5_fit_diss', 'ftv_b5_fit_savg', 'ftv_b5_fit_idm', 'ftv_b5_fit_asm', 
     'ftv_b5_fit_ent', 'ftv_b5_fit_var', 'ftv_b5_fit_corr', 'ftv_b7_fit_contrast', 'ftv_b7_fit_diss', 
     'ftv_b7_fit_savg', 'ftv_b7_fit_idm', 'ftv_b7_fit_asm', 'ftv_b7_fit_ent', 'ftv_b7_fit_var', 'ftv_b7_fit_corr', 
     'NDFI', 'ftv_nbr_fit', 'ftv_ndfi_fit', 'ftv_ndvi_fit', 'ftv_evi_fit', 
     'ftv_ndmi_fit', 'ftv_ndsi_fit', 'ftv_tcw_fit', 'ftv_tcb_fit', 'ftv_tcg_fit', 'ftv_tca_fit', 'ftv_b1_fit', 
     'ftv_b2_fit', 'ftv_b3_fit', 'ftv_b4_fit', 'ftv_b5_fit', 'ftv_b7_fit'])
    print('starting merging selected bands')
    merged_array, transform, bnames, crs = merge_selected_bands_in_memory(tif_paths, features)
    print('merging completed, adding topo vars')

    #export the transform for later export
    with open("agbd_predictions/rev_pred_cache/ensemble_model/ls_lhs_medoid_"+tile+"_NBR_3by3_dur8_biome1_val_v2_TRANSFORM.json", "w") as f:
        json.dump(transform.to_gdal(), f)

    gdal_transform = list(transform.to_gdal())

    x_pixels=merged_array.shape[2]
    y_pixels=merged_array.shape[1]
    nbands= merged_array.shape[0]

    if icc==True:
        print('AGBD prediction with ICC on Landsat')
    #         img_stack = gdal.Open(fpath)
    #         # img_stack = gdal.Open('/gpfs/data1/duncansongp/amberliang/EA_data/ancillary_vars/EAaspect_cropped.tif')
    #         img_arr = img_stack.ReadAsArray()
    #         print(type(img_arr))
        img_arr_reshape= merged_array.reshape(merged_array.shape[0], (merged_array.shape[1]*merged_array.shape[2])).transpose()

        #read in the ic_df and convert to ic_list and apply ic to lc_raster 
        ic_df0= pd.read_csv('landsat_comp/t'+t1+'_ic/ls_predictor_ic_'+ str(year+1)+"-"+str(year)+'.csv',  index_col=0)
        features_temp=[item for item in features if item not in ['slope','dem','aspect','lon_lowestmode','lat_lowestmode']]
        ic_df=ic_df0[features_temp]
        img_arr_reshape_ic = img_arr_reshape*ic_df.values.flatten()   #np.array(iclist)[None,:]
    #         print('new image stack dimension', prev_ic_img_reshape.shape)
    #         curr_image_reshape =  prev_ic_img_reshape
        print(img_arr_reshape_ic.shape)

        img_arr_reshape_ic2= img_arr_reshape_ic.transpose().reshape(nbands, -1,  x_pixels)
        if img_arr_reshape_ic2.shape == merged_array.shape:
            print(img_arr_reshape_ic2.shape, merged_array.shape)
            merged_array = img_arr_reshape_ic2    #overwrite the original img_arr with the ic-ed arr 
        #modify the dst_f to indicate it's ic-flag
        dst_f= 'icc_'+dst_f
    else:
        print('AGBD preditcion without ICC')
        merged_array =merged_array
        dst_f= dst_f

    bval = merged_array   #np.array(bandList)
    bkey = bnames  #[list(ori_bnames)[i] for i in sel_bands]
    img_dict =  {bkey[i]: bval[i] for i in range(len(bkey))}  #dict(zip(bkey, bval)) #
    # print(img_dict)
    print(len(img_dict))

    #stacking the topo vars to the image stack extent
    # Open the small raster to get its bounding box
    small_gt = gdal_transform
    xmin = small_gt[0]
    ymax = small_gt[3]
    xmax = xmin + small_gt[1] * x_pixels
    ymin = ymax + small_gt[5] * y_pixels

    # Clip the large raster using GDAL Translate
    clipped_ds = gdal.Translate('','/gpfs/data1/duncansongp/amberliang/EA_data/ancillary_vars/topo_stack_res2.tif', 
                                projWin=[xmin, ymax, xmax, ymin],format='MEM')

    topo_arrays = []
    for i in range(1,clipped_ds.RasterCount + 1):
        # Read the clipped data for the current band
        clipped_band = clipped_ds.GetRasterBand(i)
        clipped_data = clipped_band.ReadAsArray()

        # Read the small raster's mask (binary mask)
    #     small_band = small_ds.GetRasterBand(1)
        small_data = merged_array[1,:,:]#small_band.ReadAsArray()
        mask_binary = (small_data > 0).astype(int)  # Creating a binary mask

        # Mask the data: where the mask is 0, set the values to NoData
        masked_data = np.where(mask_binary == 1, clipped_data, clipped_band.GetNoDataValue())
        topo_arrays.append(masked_data)

    clipped_array= np.stack(topo_arrays, axis=0)
    print(clipped_array.shape)

    #append the clipped array to the img_dict
    src  = rasterio.open('/gpfs/data1/duncansongp/amberliang/EA_data/ancillary_vars/topo_stack_res2.tif')
    tkey = list(src.descriptions)
    tval = clipped_array
    topo_dict = {tkey[i]: tval[i] for i in range(len(tkey))}
    # print(img_dict)
    print(len(topo_dict))
    #append to img_Dict
    img_dict.update(topo_dict)
    print(len(img_dict))

    #generate lat and lon layers to extent of the image stacks
    GT = gdal_transform
    width = x_pixels
    height = y_pixels
    # Unpack the geotransform
    origin_x, pixel_width, x_rotation, origin_y, y_rotation, pixel_height = GT

    # Generate grid of pixel indices
    cols, rows = np.meshgrid(np.arange(width), np.arange(height))

    # Compute coordinates of each pixel center
    lon = origin_x + cols * pixel_width + rows * x_rotation
    lat = origin_y + cols * y_rotation + rows * pixel_height

    #append the topo and lat/lon to the band dictionaries 
    # img_arr_topo_lon= np.concatenate((img_arr_topo, np.array(lon_layer)), axis=0)
    coord_arr=np.stack([lon, lat], axis=0)
    # print(coord_arr.shape)
    ckey =['lon_lowestmode', 'lat_lowestmode']
    coord_dict = {ckey[i]: coord_arr[i] for i in range(len(ckey))}
    img_dict.update(coord_dict)
    print(len(img_dict))

     #select bands correpsonding to the model vars ###!!!![need to change]!!!!
    select_vars_del = features#np.array(ls_vars)[svi] #features[td_bool]
    print(select_vars_del)
    img_dict_reorder = {key: img_dict[key] for key in features}
    img_arr_model = list(img_dict_reorder.values())#list(img_dict_reorder)#[img_dict.get(key) for key in select_vars_del]
    print(type(img_arr_model))
    print(len(img_arr_model))

    # #convert list of arrays to ndarray
    img_arr_model2=np.stack(img_arr_model)
    print(img_arr_model2.shape)

#     #reshape the image stack  [this reshape rstack for predictions]
    img_arr_reshape = img_arr_model2.reshape(img_arr_model2.shape[0], (img_arr_model2.shape[1]*img_arr_model2.shape[2])).transpose()
    
    output_dir='agbd_predictions/rev_pred_cache/merged_ls/'  
    output_path = output_dir+'ls_lhs_medoid_'+tile+'_'+str(year)+'_NBR_3by3_dur8_biome1_val_v2_MERGED.npy'
    np.save(output_path, img_arr_reshape)

    #-----creating masks------------
        #masking out ndvi =0 and lc masked areas
    if 'ftv_ndvi_fit' in bnames:
        ndvi_ind=np.where(np.isin(bnames,'ftv_ndvi_fit'))[0].tolist()
        print('ndvi is in feature list')
        ndvi = merged_array[ndvi_ind, :,:]
        ndvi = np.squeeze(ndvi)
#         print(type(ndvi))
    else:
        ndvi, transform2, bnames2, crs2 = merge_selected_bands_in_memory(tif_paths, ['ftv_ndvi_fit'])
        ndvi = np.squeeze(ndvi)
    # print(ndvi.shape)
    
    output_dir='agbd_predictions/rev_pred_cache/lc_mask/'
    output_path = output_dir+'ls_lhs_medoid_'+tile+'_'+str(year)+'_NBR_3by3_dur8_biome1_val_v2_NDVI.npy'
    np.save(output_path, ndvi)
    

    #subset the land cover strata from GLAD product 
    lc=  gdal.Translate('','ancillary_vars/ea_water/strata_ea-0000000000-0000000000.tif', 
                                projWin=[xmin, ymax, xmax, ymin],format='MEM')

    lc_band = lc.GetRasterBand(1)
    lc_data = lc_band.ReadAsArray()

    # Read the small raster's mask (binary mask)
    small_data_lc = merged_array[1,:,:]#small_band.ReadAsArray()
    mask_binary_lc = (small_data_lc > 0).astype(int)  # Creating a binary mask

    # Mask the data: where the mask is 0, set the values to NoData
    lc_clip_reshape = np.where(mask_binary_lc == 1, lc_data, clipped_band.GetNoDataValue())
    
    print(lc_clip_reshape.shape)
    output_dir='agbd_predictions/rev_pred_cache/lc_mask/'
    output_path = output_dir+'ls_lhs_medoid_'+tile+'_'+str(year)+'_NBR_3by3_dur8_biome1_val_v2_LC.npy'
    np.save(output_path, lc_clip_reshape)
    print('merged predictor stack exported...')
    
    


def merge_agbd_uncertainty_predict(nboot, reference_y, reference_se, xtraining,year, tile,tif_paths, model, features, dst_dir,dst_f, icc=True, best_params=None): 

    #check if enselble model has been created, if not create them 
    #-----------------------load the enseble models-------------
    output_dir='agbd_predictions/rev_pred_cache/ensemble_model/' 
    emsemModelpath=output_dir+'ls_lhs_medoid_'+tile+'_NBR_3by3_dur8_biome1_val_v2_100EnsembleModels.joblib'
    
    if os.path.exists(emsemModelpath):
        print("Ensemble model exists.")
        models = load(emsemModelpath)
    else:
        print("Training Ensemble models")
        train_and_save_ensemble(nboot, reference_y, reference_se, xtraining,year, tile,tif_paths, model, features, dst_dir,dst_f, icc=True, best_params='NULL')

    #-------load the merged raster predictor stack-------
    output_dir='agbd_predictions/rev_pred_cache/merged_ls/' 
    loaded_array = np.load(output_dir+'ls_lhs_medoid_'+tile+'_'+str(year)+'_NBR_3by3_dur8_biome1_val_v2_MERGED.npy')
    
    #------loading in masks------
    mask_dir='agbd_predictions/rev_pred_cache/lc_mask/' 
    ndvi = np.load(mask_dir+'ls_lhs_medoid_'+tile+'_'+str(year)+'_NBR_3by3_dur8_biome1_val_v2_NDVI.npy')
    lc_clip_reshape=np.load(mask_dir+'ls_lhs_medoid_'+tile+'_'+str(year)+'_NBR_3by3_dur8_biome1_val_v2_LC.npy')

    y_pixels=ndvi.shape[0]
    x_pixels=ndvi.shape[1]
    nbands= loaded_array.shape[1]

    #make predictions with enselmble models
    print('making prediction for nboot iterations')
    bootstrap_predictions_ori0 = [modeli.predict(loaded_array) for modeli in models]
    
    # back_trans and bias correction
    print('prediction complete, further processing...')
    bootstrap_predictions_ori = np.array(bootstrap_predictions_ori0).T  # Shape: (n_samples, nboot)  transformed data
    bootstrap_predictions=back_trans(bootstrap_predictions_ori)*C
    print('bootstrap_predictions shape: ',bootstrap_predictions.shape)

    #combine error and derive 100 estimates of agbd at each px
    # Mean prediction and model uncertainty
    mean_prediction = np.mean(bootstrap_predictions, axis=1)
    model_uncertainty = np.std(bootstrap_predictions, axis=1)

    print('mean_prediction shape: ',mean_prediction.shape)
    print('model_uncertainty shape: ',model_uncertainty.shape)
    
    # Total uncertainty
    total_uncertainty = np.sqrt(model_uncertainty**2)

    #CI95 using the percentile)
    lower_ci =np.quantile(bootstrap_predictions,  0.025 ,axis=1) 
    upper_ci = np.quantile(bootstrap_predictions,  0.975,axis=1)
    CI95= (upper_ci-lower_ci)/2

    # perc_error, not using z score, just get the 95th inter-qauntile range[smaller]
    perc_error2=(CI95)/mean_prediction*100#still in transformed space
    
    #-------masking the predictions----
    bootstrap_predictions_T = bootstrap_predictions.T  # shape: (100, 38853120)
    agbd_rsh = bootstrap_predictions_T.reshape((nboot, y_pixels, x_pixels))  # sha
    agbd_rsh=np.where(ndvi[None, :, :]==0, np.nan, agbd_rsh)
    agbd_rsh=np.where(lc_clip_reshape[None, :, :]==16, np.nan, agbd_rsh)  
    
    #-------loading transform info-----------

    with open("agbd_predictions/rev_pred_cache/ensemble_model/ls_lhs_medoid_"+tile+"_NBR_3by3_dur8_biome1_val_v2_TRANSFORM.json") as f:
        transform_gdal = json.load(f)
    transform = Affine.from_gdal(*transform_gdal)

    #------export the nboot predictions in two chunks
    print('writing outputs in two chunks')
    profile = {
        'driver': 'GTiff',
        'dtype': agbd_rsh.dtype,
        'count': nboot*0.5,  # Number of bands
        'height': y_pixels,
        'width': x_pixels,
        'crs': 'EPSG:4326',
        'transform':transform,
        'compress': 'lzw',  # Optional: apply LZW compression
        'tiled': True,      # essential for compression efficiency
        'blockxsize': 256,
        'blockysize': 256,
        'BIGTIFF': 'YES'
    }

    if icc==True:
        dst_f= 'icc_'+dst_f


    dst_out= dst_dir+dst_f.replace(".tif", '_'+str(nboot)+'pt1.tif')

    # Write the array to the new raster file
    with rasterio.open(dst_out, 'w', **profile) as dst:
        dst.write(agbd_rsh[:int(nboot*0.5),:,:]) 

    dst_out= dst_dir+dst_f.replace(".tif", '_'+str(nboot)+'pt2.tif')

    # Write the array to the new raster file
    with rasterio.open(dst_out, 'w', **profile) as dst:
        dst.write(agbd_rsh[int(nboot*0.5):nboot,:,:]) 

print(len(args))




def main():
    tic = time.time()
    pool = Pool(processes=1) 
    t=pool.starmap(merge_agbd_export,args)
    result = pool.starmap(merge_agbd_uncertainty_predict, args)
    pool.terminate()
    pool.join()
    print(result)
    print('end')
    toc = time.time()
    print('Done in {:.4f} seconds'.format(toc-tic))

    
if __name__ == "__main__":
    main()








/home/liangm/.conda/envs/ea_work/lib/python3.7/site-packages/geopandas/_compat.py:53: UserWarning: The installed version of PyGEOS is too old (0.6 installed, 0.8 required), and thus GeoPandas will not use PyGEOS.
  UserWarning,


ModuleNotFoundError: No module named '_gdal'